# Week 5 へようこそ - エージェントフレームワーク

## Day 2: Pydantic AI

同じ週、同じ5つのステップ、新しいフレームワークです。今週の考え方全体は、ひとつのエージェントフレームワークを理解すれば他もだいたい理解できるということなので、毎日同じ5つのステップで同じエージェントを作り、その作法が響き合う様子を観察します。

1. **エージェントを作る** - モデルとシステムプロンプトを与える。
2. **実行する** - メッセージを送り、返信を受け取る。
3. **ツールを追加する** - エージェントが呼び出せる、普通の型付き関数。
4. **MCP を追加する** - 誰か他の人が書いたツールサーバーに接続する。毎回同じ方法でつなげる。
5. **ゴールを与えてループさせる** - 目標を渡し、仕事が終わるまで一歩ずつ自分で進めさせる。

ステップ1と2は、まだ単なる LLM 呼び出しです。ツールと MCP は、エージェントにできることを与えます。ステップ5でようやくエージェントらしくなります。フレームワーク自身がループを回し、ツールを選び、結果を読み、また選び直す、というのをゴールに到達するまで続けるのです。

実習プロジェクトは Day 1 と同じ SQLite の todo ボードです。ワーカーがボードから1つのゴールを取り出し、自分でステップを計画し、自分のエージェントループでその作業をこなし、各ステップにチェックを入れていきます。ボードのコード(`board.py`)は一字一句まったく同じファイルで、変わるのはそれを取り巻くフレームワークだけです。

これは今日2つ目のフレームワーク、**Pydantic AI** です。FastAPI や現代の Python データスタックの大半を支えているバリデーションライブラリ、Pydantic のチームによるものです。ツールが何を受け取り、エージェントが何を返すかを宣言すれば、フレームワークがそのフロー全体を検証し、型付けしてくれる、あの「どこもかしこも型付き」という感覚をエージェントにも持ち込んでいます。この分野でもっとも洗練された型付き出力の仕組みを持つことと、Logfire によるワンライン可観測性で知られています。Strands との違いがどれだけ少ないかに注目してください。それこそが今週伝えたいポイントです。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Pydantic AI のドキュメント</h2>
            <span style="color:#00bfff;">ドキュメントは <a href="https://pydantic.dev/docs/ai/">https://pydantic.dev/docs/ai/</a> にあります。Pydantic AI は2025年に v1 に到達し、v2 はベータ版なので、ここではバージョンを固定(1.107.0)して、古いブログ記事より最新のページを優先します。特に MCP の接続部分は最近変更されました。</span>
        </td>
    </tr>
</table>

## セットアップ

今日必要なものは2つですが、どちらも以前の週からすでに用意されています。

- **Node**。`npx` のために必要です(filesystem MCP サーバーはこれを使って動きます)。`node --version` で確認してください。
- リポジトリのルートにある `.env` の中の **`GOOGLE_API_KEY`**。今日のフレームワークは Gemini の `gemini-flash-latest` を、GeminiのOpenAI互換エンドポイント経由で使います。

Pydantic AI はリポジトリの環境に含まれているので、リポジトリのルートで通常の `uv sync` を実行すればすべてインストールされます。このノートブックを Cursor で開き、毎週使っているリポジトリ既定の **Python 3.12.12** カーネルを選んで、上から順にセルを実行してください。

最初の実行を速くするために、今のうちに一度 filesystem MCP サーバーをウォームアップしておき、動作中と表示されたらすぐに Ctrl-C で止めてください。

```bash
npx -y @modelcontextprotocol/server-filesystem .
```

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from fastmcp.client.transports import StdioTransport

load_dotenv(override=True)

## ステップ1: エージェントを作る

Pydantic AI では、エージェントは `Agent` です。モデルを表す文字列と、システムプロンプトである `instructions` を持ちます。ここではモデルを `OpenAIChatModel` として組み立て、GeminiのOpenAI互換(Chat Completions)エンドポイントを指す `OpenAIProvider` を渡しています。`GOOGLE_API_KEY` はリポジトリルートの `.env` から読み込まれます。Chat Completionsは、OpenAI互換のすべてのエンドポイントが話すAPIです。

In [ ]:
# メインモデルをOpenAIからGeminiに切り替え(GeminiのOpenAI互換エンドポイント経由)
MODEL = OpenAIChatModel(
    "gemini-flash-latest",
    provider=OpenAIProvider(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.environ["GOOGLE_API_KEY"],
    ),
)

agent = Agent(
    MODEL,
    instructions="You are a concise, friendly assistant. Reply in a single short sentence.",
)

## ステップ2: 実行する

メッセージを送り、返信を待ち、結果の `.output` を表示します。まだツールがないので、ループするものは何もなく、エージェントはただ答えるだけです。これはまだ単なる LLM 呼び出しです。同期的な `agent.run_sync(...)` もありますが、ノートブックの中では Jupyter のイベントループと協調するように `agent.run(...)` を await します。

In [ ]:
result = await agent.run("Say hello in Spanish.")
print(result.output)

## 今週のプロジェクト: SQLite の todo ボード

ワーカーは、Day 1 と同じ小さな SQLite ボード、同じ `board.py` ファイルを介して連携します。1つのファイル、1つのテーブルで、サーバーを立てる必要もありません。ワーカーには1つの**ゴール**が与えられ、それを達成するために自分自身の**ステップ**の todo をそのゴールの下に書き出し、進めるごとにチェックを入れていき、最後にゴールを完了にします。内部的にはボードは単なる辞書のリストです(ゴールの `parent_id` は None で、ステップは自分のゴールを指します)。

In [ ]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.list_todos()

`show_board()` は、Week 1 で使ったのと同じ rich スタイルで、その同じデータを綺麗に表示します。各ゴールの下にステップがインデントされて並び、完了したタスクは緑色の打ち消し線、進行中のタスクは黄色で表示されます。まだステップはありません。エージェントが計画を立てるときに自分でステップを書き出します。

In [ ]:
board.show_board()

## ステップ3: ツールを追加する

Pydantic AI でのツールは、普通の型付き Python 関数です。Pydantic が型ヒントと docstring を読み取り、JSON スキーマを自動で組み立ててくれるので、他に宣言することは何もありません。決まった書き方としては `@agent.tool` デコレーターがありますが、同じ関数を `tools=[...]` に渡しても完全に同じことができ、複数のエージェント間で関数を再利用できます。ここではその方法を使います。

ここでは3つの小さなボードツールを書きます。ボードを読む `show_todos`、ゴールをステップに分解する `plan_steps`、todo を完了にする `complete_task` です。まずは簡単なエージェントに2つだけ与えて、ボードに何があるか尋ねてみましょう。答える前に自分から `show_todos` を呼び出すことを、自分の目で確かめてください。この「決める、呼ぶ、読む、答える」というサイクルこそ、エージェントループが回り始めた瞬間です。3つのツールすべてはステップ5で一緒になります。

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [ ]:
board_agent = Agent(
    MODEL,
    instructions="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [ ]:
result = await board_agent.run("What is on the board right now, and what is its status?")
print(result.output)

## ステップ4: MCP を追加する

MCP は、単に「自分が書いていないツール」を、小さなプロトコル越しに接続したものです。今週すべてのフレームワークで使う同じ Node サーバーである filesystem リファレンスサーバーを、単一の `workspace` フォルダに限定してエージェントに与えます。これにより、エージェントはそのフォルダ内のファイルしか触れなくなります。Pydantic AI では、MCP サーバーはツールセットとして扱われます。`MCPToolset` でラップし、`toolsets=[...]` で渡し、実行の周りを `async with agent:` で囲むことで接続を開きます。

`log_file` はヌルデバイス(`Path(os.devnull)`)を指しています。これによりサーバーの起動時ログが画面に出なくなるだけでなく、もっと重要なことに、Windows 上の Jupyter カーネルからサーバーを実行できるようになります。Windows のカーネル自身の stderr には、サーバーが書き込める実際のファイルディスクリプタがないためです。Mac と Linux では、単純に出力が綺麗になるだけです。

In [ ]:
workspace = Path("workspace").resolve()   # エージェントが触れてよい唯一のフォルダ

filesystem = MCPToolset(
    StdioTransport(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
        cwd=str(workspace),  # 相対ファイル名がそこで解決されるよう、workspace内でサーバーを起動する
        log_file=Path(os.devnull),
    ), 
    init_timeout=60
)

In [ ]:
file_agent = Agent(
    MODEL,
    instructions="You can read and write files in your workspace. Use your tools to do what is asked.",
    toolsets=[filesystem],
)

In [ ]:
async with file_agent:
    result = await file_agent.run("Read notes.txt and summarize it in one short sentence.")
print(result.output)

## ステップ5: ゴールを与えてループさせる

さあ、いよいよ本番です。1つのエージェントに3つのボードツールすべてと filesystem サーバーを与え、ゴールを渡して、実行させましょう。エージェントは自分でボード上にステップを計画し、ファイルツールでそれを片付け、それぞれにチェックを入れ、作業が終わったらゴールを完了にします。これこそ、自律的に動くエージェントループです。読む、計画する、行動する、チェックする、繰り返す。ボードにステップが埋まり、それが打ち消し線で消されていく様子を観察してください。

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    MODEL,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task],
    toolsets=[filesystem],
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.claim_todo(goal_id)

async with worker:
    result = await worker.run("Please work the pending goal on the board.")
print(result.output)
board.show_board()

## 同じワーカーをターミナルから実行する

ステップ5でたった今観察した内容はすべて、このノートブックの隣にある小さなスクリプト `pydantic_worker.py` としてもパッケージ化されています。これは同じゴールを登録し、同じ3つのボードツールと filesystem MCP サーバーを使って同じエージェントを組み立て、カーネルではなくコマンドラインから同じループを実行します。このフォルダでターミナルを開いて実行してください。

```bash
uv run pydantic_worker.py
```

エージェントがステップを計画し、ゴールに取り組んでそれぞれにチェックを入れていく様子、そして完成したボードと、書き出されたスペイン語が表示されます。これは Day 5 でどのワーカーも取る形と同じです。Day 5 では、Google ADK のオーケストレーターが、フレームワークごとにこうしたワーカーを1つずつ、共有された1つのボードに対する並列サブプロセスとして起動します。

## いちばん興味深い点: 型付きで、検証済みで、観測可能

Pydantic AI は2つの考え方を軸に構築されています。今回のワーカーはそれらに頼っていませんが、知っておく価値があります。

- **型付きで検証済みの結果。** `output_type=` に Pydantic モデルを渡せば、`result.output` はその型の検証済みインスタンスとして返ってきます。型付き出力自体はどのフレームワークでもできますが、これは Pydantic AI がよく知られている部分です。
- **わずか2行での可観測性。** `logfire.configure()` と `logfire.instrument_pydantic_ai()` を書くだけで、すべてのモデルリクエスト、ツール呼び出し、検証処理が Logfire の UI 上でリアルタイムに可視化されます。これは、このフレームワークを作っているのと同じチームによるものです。これには `logfire` エクストラが必要ですが、共有環境を軽く保つためにここでは省いています。

そして、今週の他の部分と同じワンライン精神で、OpenAI 互換の任意のエンドポイントで実行するには、`OpenAIChatModel` で `OpenAIProvider` をラップしてエージェントに渡すだけです。他はすべて同じままです。

```python
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

model = OpenAIChatModel(
    "gpt-5.4-mini",
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_KEY),
)
agent = Agent(model, instructions="...")
```

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">エクササイズ</h2>
            <span style="color:#ff7800;">ボードに別のゴールを、たとえば「マドリードについての短い俳句を書いて madrid.txt に保存する」を登録し、ワーカーを再度実行してみましょう。ワーカーは適切なステップを計画し、正しいファイルツールを選べるでしょうか。実行後には <code>result.all_messages()</code> を調べて、完全なトランスクリプトを見てみましょう。ループが行ったすべてのツール呼び出しと、読み取ったすべての結果が確認できます。</span>
        </td>
    </tr>
</table>